In [1]:
# ========= DATA PROFILE + PLOTS =========
import os, re, math, json, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


warnings.filterwarnings("ignore")

DATA_DIR = r"C:\Users\kerne\Pictures\datasets"
TRAIN_INTERACTIONS = os.path.join(DATA_DIR, "train_interactions.csv")
GAMES_FILE = os.path.join(DATA_DIR, "games.csv")
PLOT_DIR = DATA_DIR
os.makedirs(PLOT_DIR, exist_ok=True)

def _detect_cols(df):
    cols = {c.lower(): c for c in df.columns}
    u = cols.get("user_id") or cols.get("user") or cols.get("userid") or cols.get("steamid")
    i = cols.get("item_id") or cols.get("item") or cols.get("game_id") or cols.get("appid") or cols.get("app_id")
    p = cols.get("playtime") or cols.get("play_time") or cols.get("playtime_forever") or cols.get("rating") or cols.get("label")
    return u, i, p

def _parse_tokens(series, label):
    # phrase-preserving split on [ , | / ; ] only
    DELIMS_RE = re.compile(r"[\,\|/;]+")
    def norm(s):
        s = str(s).strip().lower()
        s = re.sub(r"\s+", " ", s).replace("-", "_").replace(" ", "_")
        s = re.sub(r"[^a-z0-9_]+", "", s)
        return s
    all_toks = []
    missing = 0
    for raw in series.fillna(""):
        if not raw:
            missing += 1
            continue
        parts = [p for p in DELIMS_RE.split(str(raw)) if p and p.strip()]
        toks = [norm(p) for p in parts if norm(p)]
        if toks:
            all_toks.extend(toks)
        else:
            missing += 1
    return Counter(all_toks), missing

print("=== DATA PROFILE ===")
inter = pd.read_csv(TRAIN_INTERACTIONS, low_memory=False)
u_col, i_col, p_col = _detect_cols(inter)
inter = inter.rename(columns={u_col:"user", i_col:"item"})
if p_col:
    inter["playtime"] = inter[p_col].astype(float)
else:
    inter["playtime"] = 1.0
inter = inter.dropna(subset=["user","item"])
inter["user"] = inter["user"].astype(int)
inter["item"] = inter["item"].astype(int)
inter = inter.drop_duplicates(["user","item"], keep="last")

n_int = len(inter)
n_users = inter["user"].nunique()
n_items = inter["item"].nunique()
density = 100.0 * n_int / (n_users * n_items)
avg_hist = inter.groupby("user")["item"].nunique().mean()
avg_pop = inter.groupby("item")["user"].nunique().mean()

print(f"Interactions: {n_int:,} | Users: {n_users:,} | Items: {n_items:,} | Density: {density:.4f}%")
print(f"Avg. items per user: {avg_hist:.2f} | Avg. users per item: {avg_pop:.2f}")

# Cold-start stats (consistent with later random_split convention: 80/10/10)
from libreco.data import random_split
tmp_train, tmp_eval, tmp_test = random_split(inter[["user","item"]], multi_ratios=[0.8, 0.1, 0.1], seed=42)
u_train, i_train = set(tmp_train["user"]), set(tmp_train["item"])
u_test, i_test = set(tmp_test["user"]), set(tmp_test["item"])
cold_users = len([u for u in u_test if u not in u_train])
cold_items = len([it for it in i_test if it not in i_train])
print(f"Cold users in test: {cold_users:,} / {len(u_test):,} | Cold items in test: {cold_items:,} / {len(i_test):,}")

# Token coverage from games.csv
games = pd.read_csv(GAMES_FILE, low_memory=False)
cg = {c.lower(): c for c in games.columns}
g_item = cg.get('item_id') or cg.get('appid') or cg.get('app_id') or cg.get('id')
g_tags = cg.get('tags') or cg.get('tag')
g_genres = cg.get('genres') or cg.get('genre')
games = games.rename(columns={g_item:"item", g_tags:"tags", g_genres:"genres"})
games["item"] = games["item"].astype(int)

in_catalog = set(inter["item"])
games = games[games["item"].isin(in_catalog)]
tag_counts, miss_tags = _parse_tokens(games["tags"] if "tags" in games else pd.Series([]), "tags")
genre_counts, miss_gen = _parse_tokens(games["genres"] if "genres" in games else pd.Series([]), "genres")
unique_tokens = len(set(tag_counts.keys()) | set(genre_counts.keys()))
print(f"Items with missing tags: {miss_tags:,} | missing genres: {miss_gen:,} | unique tokens (tags+genres): {unique_tokens:,}")

# ========== Plots ==========
# 1) User activity (items per user): long-tail
hist_sizes = inter.groupby("user")["item"].nunique()
plt.figure(figsize=(7,4))
plt.hist(hist_sizes.values, bins=100)
plt.yscale("log")
plt.xlabel("Items per user")
plt.ylabel("Count (log scale)")
plt.title("User activity distribution")
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "profile_user_activity.png")); plt.close()

# 2) Item popularity (users per item): long-tail
item_pop = inter.groupby("item")["user"].nunique()
plt.figure(figsize=(7,4))
plt.hist(item_pop.values, bins=100)
plt.yscale("log")
plt.xlabel("Users per item")
plt.ylabel("Count (log scale)")
plt.title("Item popularity distribution")
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "profile_item_popularity.png")); plt.close()

# 3) Playtime distribution (per interaction)
plt.figure(figsize=(7,4))
plt.hist(np.clip(inter["playtime"].values, 0, np.percentile(inter["playtime"].values, 99)), bins=100)
plt.xlabel("Playtime (clipped at 99th pct)")
plt.ylabel("Count")
plt.title("Playtime distribution")
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "profile_playtime.png")); plt.close()

# 4) Top tokens bar
top_tokens = (tag_counts + genre_counts).most_common(20)
if top_tokens:
    labels, counts = zip(*top_tokens)
    plt.figure(figsize=(8,5))
    plt.bar(range(len(labels)), counts)
    plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
    plt.ylabel("Doc frequency (#items having token)")
    plt.title("Top 20 tokens (tags+genres)")
    plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "profile_top_tokens.png")); plt.close()

print("Saved: profile_user_activity.png, profile_item_popularity.png, profile_playtime.png, profile_top_tokens.png")

=== DATA PROFILE ===
Interactions: 2,267,389 | Users: 54,315 | Items: 8,368 | Density: 0.4989%
Avg. items per user: 41.75 | Avg. users per item: 270.96
Cold users in test: 0 / 42,071 | Cold items in test: 0 / 5,760
Items with missing tags: 4 | missing genres: 404 | unique tokens (tags+genres): 341
Saved: profile_user_activity.png, profile_item_popularity.png, profile_playtime.png, profile_top_tokens.png


In [1]:
import os, re, math, warnings, time
from collections import defaultdict, Counter
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import random

warnings.filterwarnings("ignore")

# LibRecommender
from libreco.data import random_split, DatasetPure
from libreco.algorithms import SVDpp, ItemCF
# sklearn
from sklearn.feature_extraction.text import TfidfVectorizer

# HERE I AM DOING: Importing all necessary libraries for the project.
# I REPEAT: The first block imports standard Python libraries like `os`, `re`, and `math` for basic operations, `collections` and `itertools` for data structures and efficient looping, `numpy` and `pandas` for numerical operations and data manipulation, `matplotlib.pyplot` for plotting, and `tqdm` for showing progress bars.
# The second block imports specialized libraries: `libreco` for building collaborative filtering models like SVD++ and ItemCF, and `sklearn` for creating a TF-IDF vectorizer, which is a key component for the content-based part of our hybrid recommender.


Instructions for updating:
non-resource variables are not supported in the long term


In [4]:
# -------------------- SETTINGS --------------------
DATA_DIR = r"C:\Users\GSI\Pictures\DATASETS"
TRAIN_INTERACTIONS = os.path.join(DATA_DIR, "train_interactions.csv")
TEST_IN_FILE = os.path.join(DATA_DIR, "test_interactions_in.csv")
GAMES_FILE = os.path.join(DATA_DIR, "games.csv")

OUTPUT_TOP20 = os.path.join(DATA_DIR, "top20_recommendations.csv")
EVAL_SUMMARY_OUT = os.path.join(DATA_DIR, "evaluation_summary.csv")
PLOT_DIR = DATA_DIR

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

K = 20
K_LIST = [5, 10, 20]
ALPHAS_GRID = np.arange(0.0, 1.0, 0.05)

# gating / candidate settings
ADAPT_M, ADAPT_S = 5, 2
CAND_CF = 500
CAND_CB = 400
RRF_K = 60         # RRF constant
MIN_TAG_ITEMS = 3
MIN_PROFILE_NORM = 0.10

CPU_CORES = max(1, os.cpu_count() or 1)
TF_SESS_CONFIG = {"intra_op_parallelism_threads": CPU_CORES, "inter_op_parallelism_threads": CPU_CORES}

# HERE I AM DOING: Setting up all the global configuration parameters for the script.
# I REPEAT: This section defines file paths for input and output data, ensuring the script can find the data and save results. It sets a random **seed** for reproducibility, meaning the same random operations will produce the same results on different runs. It also defines key hyperparameters for the recommendation system:
# - `K`: The number of recommendations to generate (Top-20).
# - `ALPHAS_GRID`: A range of values from 0.0 to 1.0 to test for blending collaborative filtering (CF) and content-based (CB) scores. An alpha of 1.0 means 100% CF, and 0.0 means 100% CB.
# - `CAND_CF` and `CAND_CB`: The number of candidate items to retrieve from each model (CF and CB) before blending, which is a performance optimization.
# - `RRF_K`: A constant for the Reciprocal Rank Fusion (RRF) blending algorithm, which is used to combine the scores from different models.
# - `MIN_TAG_ITEMS` and `MIN_PROFILE_NORM`: These are used for a dynamic, "adaptive" blending strategy. If a user's content profile is weak (fewer tagged items or low profile vector norm), the script will lean more heavily on the CF model.
# - `CPU_CORES`: Configures TensorFlow to use all available CPU cores for training, speeding up the process.



In [5]:
# -------------------- HELPERS --------------------
def load_csv(path):
    print(f"Loading {path} ...")
    df = pd.read_csv(path, low_memory=False)
    print(f"Loaded {len(df):,} rows, columns: {list(df.columns)[:8]}{'...' if len(df.columns)>8 else ''}")
    return df

def detect_columns(df):
    cols = [c.lower() for c in df.columns]
    def pick(cands):
        for cand in cands:
            if cand in cols: return df.columns[cols.index(cand)]
        return None
    return (
        pick(("user_id","user","userid","user id","steamid")),
        pick(("item_id","itemid","item","game_id","appid","app_id","id")),
        pick(("playtime","play_time","play_minutes","playtime_forever","rating","label"))
    )

DELIMS_RE = re.compile(r"[\,\|/;]+")
PHRASE_SYNONYMS = {
    "fps":"first_person_shooter", "tps":"third_person_shooter",
    "rpg":"role_playing", "arpg":"action_rpg",
    "roguelike":"rogue_like", "rogue_like":"rogue_like", "rogue_lite":"rogue_lite",
}
def normalize_phrase(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s).replace("-", "_").replace(" ", "_")
    s = re.sub(r"[^a-z0-9_]+", "", s)
    return PHRASE_SYNONYMS.get(s, s) if s else ""
def parse_tags_or_genres(raw: str) -> list:
    if pd.isna(raw): return []
    parts = [p for p in DELIMS_RE.split(str(raw)) if p and p.strip()]
    toks, seen = [], set()
    for p in parts:
        tok = normalize_phrase(p)
        if tok and tok not in seen:
            seen.add(tok); toks.append(tok)
    return toks

def sigmoid(x): return 1.0/(1.0+np.exp(-x))

# robust top-k selector (expects allow_idx to be positions into all_items array)
def topk_from_scores(scores, allow_idx, k: int):
    scores = np.asarray(scores).reshape(-1)
    allow_idx = np.asarray(allow_idx, dtype=int)
    n_items = len(all_items)

    # normalize shapes
    if scores.size != n_items:
        if scores.size == 1:
            scores = np.full(n_items, float(scores[0]), dtype=float)
        elif scores.size == allow_idx.size and allow_idx.size > 0:
            tmp = np.full(n_items, -np.inf, dtype=float)
            tmp[allow_idx] = scores
            scores = tmp
        elif scores.ravel().size == n_items:
            scores = scores.ravel()
        else:
            scores = np.full(n_items, -np.inf, dtype=float)

    if allow_idx.size == 0:
        return np.array([], dtype=int)

    take = min(k, allow_idx.size)
    if allow_idx.size == 1:
        return allow_idx.copy()

    part = np.argpartition(-scores[allow_idx], take - 1)[:take]
    cand = allow_idx[part]
    order = cand[np.argsort(-scores[cand])]
    return order[:k]

def recall_ndcg_hr_at_k(topk_items, gt_set, k=K):
    if not gt_set: return 0.0, 0.0, 0.0
    hits = [1 if int(it) in gt_set else 0 for it in topk_items]
    recall = sum(hits) / float(len(gt_set))
    hr = 1.0 if sum(hits) > 0 else 0.0
    dcg = sum(rel / math.log2(r+2) for r, rel in enumerate(hits))
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, min(len(gt_set), k) + 1))
    ndcg = dcg / idcg if idcg > 0 else 0.0
    return recall, ndcg, hr

def pairwise_cosine_from_tfidf(rows):
    # rows: csr matrix [k x F]
    if rows.shape[0] < 2: return 0.0
    sim = rows @ rows.T
    sim = sim.toarray()
    iu = np.triu_indices(sim.shape[0], k=1)
    return float(1.0 - sim[iu].mean())  # diversity = 1 - mean similarity

# RRF utility: convert a raw score vector into reciprocal-rank fusion scores
def score_to_rrf(scores, k_rrf=RRF_K):
    n = len(scores)
    # treat -inf or nan robustly
    s = np.asarray(scores).astype(float)
    # if all equal, return uniform small values
    if np.all(np.isfinite(s)) and (s.max() == s.min()):
        return np.ones(n) / (k_rrf + 1.0)
    # argsort descending
    order = np.argsort(-s, kind="mergesort")
    rrf = np.zeros(n, dtype=float)
    for r, pos in enumerate(order):
        rrf[pos] = 1.0 / (k_rrf + r + 1.0)
    # normalize to [0,1] scale
    if rrf.max() > 0:
        rrf = rrf / rrf.max()
    return rrf

def rrf_blend_from_scores(allow_idx, cf_scores, cb_scores, alpha):
    # create full-length RRF scores for each signal then blend
    cf_rrf = score_to_rrf(cf_scores)
    cb_rrf = score_to_rrf(cb_scores)
    blended = alpha * cf_rrf + (1.0 - alpha) * cb_rrf
    return blended

def conservative_alpha(u, base_alpha):
    # adapt alpha per user: if user's content profile is weak, shift to CF-heavy
    prof_norm = user_prof_norm.get(int(u), 0.0)
    tagged_cnt = user_tagged_counts.get(int(u), 0)
    if prof_norm <= 0 or tagged_cnt < MIN_TAG_ITEMS:
        # make CF stronger: increase alpha toward 0.9..1.0
        bump = 0.5 + 0.5 * (MIN_TAG_ITEMS - min(tagged_cnt, MIN_TAG_ITEMS)) / float(max(1, MIN_TAG_ITEMS))
        return float(min(1.0, base_alpha + (1.0 - base_alpha) * bump))
    # otherwise slightly reduce CF weight when profile is strong
    # scale by sigmoid of normalized profile
    factor = sigmoid((prof_norm - MIN_PROFILE_NORM) * ADAPT_S)
    return float(base_alpha * (0.5 + 0.5 * factor))

# HERE I AM DOING: Defining a series of helper functions that handle data loading, normalization, score calculation, and evaluation.
# I REPEAT: This section contains the core logic for the recommendation pipeline's custom parts.
# - `load_csv`: A simple function to load a CSV file and print a summary.
# - `detect_columns`: This is a robust function that automatically identifies the user, item, and playtime columns from a dataframe, even if they have different names (e.g., `user_id`, `steamid`, `user`). This makes the script more flexible.
# - `normalize_phrase` and `parse_tags_or_genres`: These functions are crucial for the content-based component. They clean and standardize text data (tags/genres) by converting to lowercase, replacing special characters, and mapping synonyms (e.g., "FPS" to "first_person_shooter") to ensure consistency.
# - `sigmoid`: A standard activation function used here for the adaptive alpha calculation.
# - `topk_from_scores`: This is a critical utility. Given a list of scores for all items, it efficiently finds the top-K items, while ensuring that previously consumed items are not recommended. It uses `np.argpartition` for a fast, partial sort.
# - `recall_ndcg_hr_at_k`: This function calculates three standard recommendation metrics: **Recall** (how many ground-truth items were found), **NDCG** (Normalized Discounted Cumulative Gain, which weights hits at the top of the list more heavily), and **Hit Rate** (did we get at least one ground-truth item?).
# - `pairwise_cosine_from_tfidf`: Measures the diversity of a list of recommendations by calculating the average cosine similarity between all pairs of items in the list. A lower score means higher diversity (more different items).
# - `score_to_rrf` and `rrf_blend_from_scores`: These are the heart of the **hybrid blending**. `score_to_rrf` converts raw scores into a "reciprocal rank" score, where the highest-ranked item gets a score of $1/(K_{rrf}+1)$, the second gets $1/(K_{rrf}+2)$, and so on. `rrf_blend_from_scores` then combines these RRF scores from the CF and CB models using a weighted average (`alpha`). This method is robust to different score ranges from the underlying models.
# - `conservative_alpha`: This function implements the adaptive blending logic. It checks a user's content profile strength (how many tagged items they have interacted with and the norm of their profile vector). If the profile is weak, it increases the `alpha` value to give more weight to the collaborative filtering model, which doesn't rely on content features.

# -------------------- STEP 1: Load & clean --------------------
print("\n=== STEP 1: Load datasets ===")
inter_df = load_csv(TRAIN_INTERACTIONS)
u_col, i_col, p_col = detect_columns(inter_df)
print(f"Detected -> user: {u_col}, item: {i_col}, playtime/label: {p_col}")

inter_df = inter_df.rename(columns={u_col:"user", i_col:"item"})
inter_df = inter_df.dropna(subset=["user","item"])
inter_df["user"] = inter_df["user"].astype(int)
inter_df["item"] = inter_df["item"].astype(int)
if p_col:
    inter_df["playtime"] = inter_df[p_col].astype(float)
else:
    inter_df["playtime"] = 1.0
inter_df["label"] = (inter_df["playtime"] > 0).astype(int)
inter_df = inter_df.drop_duplicates(["user","item"], keep="last")
print(f"After cleaning: {len(inter_df):,} rows, users={inter_df['user'].nunique():,}, items={inter_df['item'].nunique():,}")

games_df = pd.read_csv(GAMES_FILE, low_memory=False)
print(f"Loaded games.csv with {len(games_df):,} rows")

# Optional test_in (for final inference)
if os.path.exists(TEST_IN_FILE):
    test_in_df = pd.read_csv(TEST_IN_FILE, low_memory=False)
    tu, ti, _ = detect_columns(test_in_df)
    test_in_df = test_in_df.rename(columns={tu:'user', ti:'item'})
    test_in_df['user'] = test_in_df['user'].astype(int)
    test_in_df['item'] = test_in_df['item'].astype(int)
else:
    test_in_df = None

# HERE I AM DOING: Loading and preprocessing the core interaction and item metadata datasets.
# I REPEAT: The script loads `train_interactions.csv` and `games.csv` into pandas DataFrames. It uses the `detect_columns` helper to standardize column names to 'user', 'item', and 'playtime', making the rest of the code cleaner. It then performs crucial cleaning steps:
# - Dropping rows with missing user or item IDs.
# - Converting user and item IDs to integers.
# - If playtime data exists, it's used; otherwise, a default playtime of 1.0 is assigned.
# - A binary `label` (0 or 1) is created based on whether playtime is greater than zero, which is needed for ranking models.
# - Duplicate user-item pairs are removed, keeping only the last interaction.
# Finally, it checks for an optional `test_interactions_in.csv` file, which would be used for generating final recommendations.



=== STEP 1: Load datasets ===
Loading C:\Users\GSI\Pictures\DATASETS\train_interactions.csv ...
Loaded 2,293,985 rows, columns: ['user_id', 'item_id', 'item_name', 'playtime']
Detected -> user: user_id, item: item_id, playtime/label: playtime
After cleaning: 2,267,389 rows, users=54,315, items=8,368
Loaded games.csv with 8,523 rows


In [6]:
# -------------------- STEP 2: random split --------------------
print("\n=== STEP 2: random_split (0.8 / 0.1 / 0.1) ===")
train_df, eval_df, test_df = random_split(inter_df[['user','item','label']], multi_ratios=[0.8,0.1,0.1], seed=SEED)
print(f"Split -> train={len(train_df):,}, eval={len(eval_df):,}, test={len(test_df):,}")

# HERE I AM DOING: Splitting the data into training, evaluation, and test sets.
# I REPEAT: I am using the `random_split` function from `libreco` to divide the cleaned interaction data into three sets.
# - The **training set (80%)** is used to train our recommendation models.
# - The **evaluation set (10%)** is used for hyperparameter tuning, specifically for finding the best settings for the SVD++ model.
# - The **test set (10%)** is held out completely and used only at the very end to evaluate the final performance of all models (SVD++, Content, Popularity, ItemCF, Hybrid). This ensures an unbiased final performance metric.



=== STEP 2: random_split (0.8 / 0.1 / 0.1) ===
Split -> train=1,813,911, eval=226,292, test=226,243


In [8]:
# -------------------- STEP 3: DatasetPure (for initial tune) --------------------
print("\n=== STEP 3: Build DatasetPure ===")
train_data, data_info = DatasetPure.build_trainset(train_df)
eval_data = DatasetPure.build_evalset(eval_df)
test_data = DatasetPure.build_testset(test_df)
all_items = np.array(data_info.item_unique_vals, dtype=int)
all_users = np.array(data_info.user_unique_vals, dtype=int)
item2pos = {it: i for i, it in enumerate(all_items)}
# Build original-user -> set(original-items) mapping (safe, original ids)
id2user = data_info.id2user
id2item = data_info.id2item
user2hist_orig = {}
for inner_u, inner_item_list in data_info.user_consumed.items():
    orig_u = id2user[inner_u]
    user2hist_orig[orig_u] = set(id2item[i] for i in inner_item_list)

# Popularity from train (initial)
pop_counts = train_df['item'].value_counts().to_dict()
pop_score = np.array([pop_counts.get(int(it), 0) for it in all_items], dtype=float)
pop_score = pop_score / (pop_score.max() or 1.0)
# HERE I AM DOING: Preparing the data for the `libreco` models and creating global mappings.
# I REPEAT: I am using `libreco`'s `DatasetPure.build_trainset` function to transform the pandas DataFrames into a format optimized for its models. This step creates a `data_info` object, which is a critical piece of metadata. It contains mappings between original user/item IDs (e.g., `steamid`s) and the `libreco`'s internal integer IDs.
# I am also extracting key information from `data_info` into global variables like `all_items`, `all_users`, and `item2pos`. The `user2hist_orig` dictionary is created to map original user IDs to the set of items they have already interacted with. This mapping is essential for filtering out consumed items when making recommendations to avoid recommending something a user has already "bought" or "played".
# Finally, I am calculating a basic popularity score for each item based on its frequency in the training data, which will serve as a simple baseline model.



=== STEP 3: Build DatasetPure ===


In [9]:
# -------------------- STEP 4: Content TF-IDF (phrase-preserving) --------------------
print("\n=== STEP 4: Content TF-IDF (tags+genres) ===")
cg = {c.lower(): c for c in games_df.columns}
g_item = cg.get('item_id') or cg.get('appid') or cg.get('app_id') or cg.get('id')
g_tags = cg.get('tags') or cg.get('tag')
g_genre = cg.get('genres') or cg.get('genre')

meta = games_df[[g_item] + ([g_tags] if g_tags else []) + ([g_genre] if g_genre else [])].copy()
meta = meta.rename(columns={g_item:'item', g_tags:'tags', g_genre:'genres'})
meta['item'] = meta['item'].astype(int)
meta = meta[meta['item'].isin(all_items)].copy()
meta['toks_tags']    = meta['tags'].apply(parse_tags_or_genres)    if 'tags'    in meta.columns else [[]]
meta['toks_genres'] = meta['genres'].apply(parse_tags_or_genres) if 'genres' in meta.columns else [[]]
meta_map = meta.set_index('item')

content_docs = []
miss_tags = miss_gen = 0
for it in all_items:
    if it in meta_map.index:
        r = meta_map.loc[it]
        tt = r['toks_tags'] if 'toks_tags' in meta_map.columns else []
        gg = r['toks_genres'] if 'toks_genres' in meta_map.columns else []
    else:
        tt, gg = [], []
    if not tt: miss_tags += 1
    if not gg: miss_gen  += 1
    prefixed = [f"t:{t}" for t in tt] + [f"g:{g}" for g in gg]
    content_docs.append(" ".join(prefixed))
print(f"Items missing tags: {miss_tags:,} / {len(all_items):,}; missing genres: {miss_gen:,} / {len(all_items):,}")

vec = TfidfVectorizer(token_pattern=r"(?u)\b[\w:]+\b", lowercase=False, sublinear_tf=True, use_idf=True, norm='l2', min_df=2)
item_content_tfidf = vec.fit_transform(content_docs).tocsr()
print(f"TF-IDF shape: {item_content_tfidf.shape}")

# Early plot: content coverage
nz = item_content_tfidf.getnnz(axis=1)
if hasattr(nz, 'A1'):
    nz = nz.A1
plt.figure(figsize=(7,4)); plt.hist(nz, bins=50); plt.xlabel("#content tokens per item"); plt.ylabel("Count"); plt.title("Content coverage per item")
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "content_coverage_hist.png")); plt.close()

# HERE I AM DOING: Building a content-based representation of items using TF-IDF.
# I REPEAT: This step is about preparing the item data (`games.csv`) for the content-based recommender.
# - I first select the item ID, tags, and genres columns from the `games` DataFrame.
# - Using the `parse_tags_or_genres` helper function, I clean and normalize the tag and genre strings into a list of standardized tokens (e.g., 'first_person_shooter').
# - I then create a "document" for each item, where the document is a single string of all its cleaned tags and genres. A prefix (`t:` or `g:`) is added to each token to ensure that a tag "rpg" and a genre "rpg" are treated as distinct features.
# - Finally, I use `TfidfVectorizer` from `sklearn` to convert these text documents into a sparse matrix (`item_content_tfidf`). Each row represents an item, and each column represents a content feature (tag/genre). The value in each cell is a TF-IDF score, which measures how important a specific feature is to a specific item. This matrix is the heart of our content-based model. I'm also plotting a histogram to visualize how many content features each item has, which helps understand the data's quality.

# -------------------- Evaluation helpers (consistent) --------------------
# Exclude consumed items: use user2hist_orig which maps original user->set(original item ids)
def exclude_consumed_idx(u_orig):
    consumed = user2hist_orig.get(int(u_orig), set())
    allow = np.array([i for i, it in enumerate(all_items) if int(it) not in consumed], dtype=int)
    return allow

# general eval for libreco models that implement recommend_user(inner_id=False)
def recommend_eval_by_model_recommend_user(model, test_truth_map, k=K, users_sample=None):
    users = [u for u in test_truth_map.keys() if int(u) in model.data_info.user2id]
    if users_sample is not None and len(users) > users_sample:
        random.seed(SEED)
        users = random.sample(users, users_sample)

    recalls, ndcgs, hrs = [], [], []
    rec_items_all = []
    for u in tqdm(users, desc=f"Eval {model.__class__.__name__} (top-{k})"):
        gt = set(test_truth_map.get(u, []))
        if not gt: continue
        recs = model.recommend_user(int(u), n_rec=k, filter_consumed=True, inner_id=False)
        if isinstance(recs, dict):
            topk = recs.get(int(u), np.array([], dtype=int))
        else:
            topk = np.asarray(recs, dtype=int)
        if topk.size == 0:
            continue
        r, n, hr = recall_ndcg_hr_at_k(topk, gt, k)
        recalls.append(r); ndcgs.append(n); hrs.append(hr)
        rec_items_all.extend(topk.tolist())
    return {
        "model": model.__class__.__name__,
        "recall": float(np.mean(recalls) if recalls else 0.0),
        "ndcg": float(np.mean(ndcgs) if ndcgs else 0.0),
        "hitrate": float(np.mean(hrs) if hrs else 0.0),
        "coverage": len(set(rec_items_all)) / float(len(all_items)) if rec_items_all else 0.0,
        "diversity": None,
        "pop_bias": None
    }

# score-based eval: returns metrics + coverage + diversity + pop_bias
def eval_model_scores(get_scores_fn, label, k=K, users_sample=None):
    users = list(test_truth.keys())
    if users_sample is not None and len(users) > users_sample:
        random.seed(SEED)
        users = random.sample(users, users_sample)

    recalls, ndcgs, hrs = [], [], []
    rec_items_all = []
    diversity_list = []
    pop_percentiles = []

    # popularity percentile mapping for items
    item_pop = train_plus_eval_df.groupby("item")["user"].count()
    # build percentile per item (higher popularity -> higher percentile)
    all_freqs = np.array([item_pop.get(int(it), 0) for it in all_items], dtype=float)
    # handle zeros
    if all_freqs.max() > 0:
        pop_rank = np.argsort(np.argsort(-all_freqs))  # 0 = most popular
        pop_percentile = 1.0 - (pop_rank / float(len(all_freqs)-1 + 1e-12))
    else:
        pop_percentile = np.zeros_like(all_freqs)

    for u in tqdm(users, desc=f"Eval scores {label} (top-{k})"):
        gt = set(test_truth.get(u, []))
        if not gt: continue
        allow = exclude_consumed_idx(u)
        if allow.size == 0: continue
        scores = get_scores_fn(u)
        topk_idx_sorted = topk_from_scores(scores, allow, k)
        if topk_idx_sorted.size == 0: continue
        topk_items = all_items[topk_idx_sorted]
        r, n, hr = recall_ndcg_hr_at_k(topk_items, gt, k)
        recalls.append(r); ndcgs.append(n); hrs.append(hr)
        rec_items_all.extend(topk_items.tolist())

        # diversity: compute pairwise diversity for this user's list (if TF-IDF available)
        try:
            rows = item_content_tfidf[topk_idx_sorted]
            diversity_list.append(pairwise_cosine_from_tfidf(rows))
        except Exception:
            pass
        # pop bias
        pp = pop_percentile[topk_idx_sorted].mean() if topk_idx_sorted.size > 0 else 0.0
        pop_percentiles.append(pp)

    return {
        "model": label,
        "recall": float(np.mean(recalls) if recalls else 0.0),
        "ndcg": float(np.mean(ndcgs) if ndcgs else 0.0),
        "hitrate": float(np.mean(hrs) if hrs else 0.0),
        "coverage": len(set(rec_items_all)) / float(len(all_items)) if rec_items_all else 0.0,
        "diversity": float(np.mean(diversity_list) if diversity_list else 0.0),
        "pop_bias": float(np.mean(pop_percentiles) if pop_percentiles else 0.0),
    }

# HERE I AM DOING: Defining the evaluation functions. This is where I can measure how well the different models perform.
# I REPEAT: This section contains two main evaluation functions to measure model quality.
# - `recommend_eval_by_model_recommend_user`: This function is designed to evaluate models that have a built-in `recommend_user` method, like SVD++ and ItemCF from `libreco`. It iterates through a list of users, calls the model's recommendation function, and then calculates the evaluation metrics (`recall`, `ndcg`, `hitrate`) by comparing the model's output to the ground-truth items. It doesn't calculate diversity or popularity bias because those metrics require access to the raw scores, not just the final ranked list.
# - `eval_model_scores`: This is a more comprehensive function designed for models where we can get a score for every single item. This includes our content-based, popularity, and hybrid models. It's more complex because it first gets all the scores, then uses the `topk_from_scores` helper to find the best recommendations (while excluding consumed items). This function is crucial because it can calculate **diversity** (using the TF-IDF matrix) and **popularity bias** (by comparing the popularity of recommended items to the overall item distribution), which are important metrics for a well-rounded recommender system.



=== STEP 4: Content TF-IDF (tags+genres) ===
Items missing tags: 4 / 8,184; missing genres: 385 / 8,184
TF-IDF shape: (8184, 356)


In [ ]:
# -------------------- STEP 5: Hyperparameter Tuning for SVD++ --------------------
print("\n=== STEP 5: Hyperparameter Tuning for SVD++ (recommend_user-based eval) ===")
eval_truth = eval_df.groupby('user')['item'].apply(list).to_dict()

param_grid = {
    "embed_size": [32, 64],          # lower + higher dimensional embeddings
    "lr": [1e-3, 5e-4],            # typical safe learning rates
    "num_neg": [3, 5],             # number of negatives per positive
    "n_epochs": [10, 20],            # short vs slightly longer training
}
best_score = -np.inf
best_params = None
EVAL_SAMPLE_USERS = min(2000, len(eval_truth))
for embed_size, lr, num_neg, n_epochs in product(param_grid["embed_size"], param_grid["lr"], param_grid["num_neg"], param_grid["n_epochs"]):
    print(f"\n>>> SVD++ trial: embed={embed_size}, lr={lr}, num_neg={num_neg}, epochs={n_epochs}")
    try:
        import tensorflow as tf
        tf.compat.v1.reset_default_graph()
    except Exception:
        pass

    m = SVDpp(
        task="ranking",
        data_info=data_info,
        embed_size=embed_size,
        n_epochs=n_epochs,
        lr=lr,
        batch_size=2048,
        num_neg=num_neg,
        tf_sess_config=TF_SESS_CONFIG,
        seed=SEED,
    )
    m.fit(train_data, neg_sampling=True, verbose=0, shuffle=True)
    r = recommend_eval_by_model_recommend_user(m, eval_truth, k=K, users_sample=EVAL_SAMPLE_USERS)
    print(f"Trial -> recall@{K}: {r['recall']:.4f}, ndcg@{K}: {r['ndcg']:.4f}")
    if r["ndcg"] > best_score:
        best_score = r["ndcg"]
        best_params = (embed_size, lr, num_neg, n_epochs)

print(f"\nBest SVD++ params (by ndcg@{K} on eval sample): {best_params}, ndcg={best_score:.4f}")

# -------------------- Retrain final SVD++ on train+eval --------------------
print("\nRetraining SVD++ on train+eval with best params ...")
train_plus_eval_df = pd.concat([train_df, eval_df], ignore_index=True)
train_data_final, data_info_final = DatasetPure.build_trainset(train_plus_eval_df)

try:
    import tensorflow as tf
    tf.compat.v1.reset_default_graph()
except Exception:
    pass

svdpp = SVDpp(
    task="ranking",
    data_info=data_info_final,
    embed_size=best_params[0],
    n_epochs=best_params[3],
    lr=best_params[1],
    batch_size=2048,
    num_neg=best_params[2],
    tf_sess_config=TF_SESS_CONFIG,
    seed=SEED,
)
svdpp.fit(train_data_final, neg_sampling=True, verbose=0, shuffle=True)

# update global mappings to final
data_info = data_info_final
all_items = np.array(data_info.item_unique_vals, dtype=int)
all_users = np.array(data_info.user_unique_vals, dtype=int)
item2pos = {it: i for i, it in enumerate(all_items)}
id2user = data_info.id2user
id2item = data_info.id2item

# rebuild user2hist_orig using final data_info (keys=original users)
user2hist_orig = {}
for inner_u, inner_item_list in data_info.user_consumed.items():
    orig_u = id2user[inner_u]
    user2hist_orig[orig_u] = set(id2item[i] for i in inner_item_list)

# rebuild TF-IDF over final all_items (if desired)
meta = games_df[[g_item] + ([g_tags] if g_tags else []) + ([g_genre] if g_genre else [])].copy()
meta = meta.rename(columns={g_item:'item', g_tags:'tags', g_genre:'genres'})
meta['item'] = meta['item'].astype(int)
meta = meta[meta['item'].isin(all_items)].copy()
meta['toks_tags']    = meta['tags'].apply(parse_tags_or_genres)    if 'tags'    in meta.columns else [[]]
meta['toks_genres'] = meta['genres'].apply(parse_tags_or_genres) if 'genres' in meta.columns else [[]]
meta_map = meta.set_index('item')
content_docs = []
for it in all_items:
    if it in meta_map.index:
        r = meta_map.loc[it]
        tt = r['toks_tags'] if 'toks_tags' in meta_map.columns else []
        gg = r['toks_genres'] if 'toks_genres' in meta_map.columns else []
    else:
        tt, gg = [], []
    prefixed = [f"t:{t}" for t in tt] + [f"g:{g}" for g in gg]
    content_docs.append(" ".join(prefixed))
vec = TfidfVectorizer(token_pattern=r"(?u)\b[\w:]+\b", lowercase=False, sublinear_tf=True, use_idf=True, norm='l2', min_df=2)
item_content_tfidf = vec.fit_transform(content_docs).tocsr()

# recompute popularity on train_plus_eval
pop_counts = train_plus_eval_df['item'].value_counts().to_dict()
pop_score = np.array([pop_counts.get(int(it), 0) for it in all_items], dtype=float)
pop_score = pop_score / (pop_score.max() or 1.0)

# HERE I AM DOING: Training the SVD++ model and then re-training it with the best hyperparameters.
# I REPEAT: This section performs two key sub-steps.
# First, I am performing a **hyperparameter search** for the SVD++ model. I loop through a predefined grid of parameters for embedding size, learning rate, number of negative samples, and epochs. For each combination, I train an SVD++ model on the `train_df` and evaluate its performance (specifically `ndcg`) on the `eval_df`. This helps me find the best-performing set of parameters without touching the final test set.
# Second, once the best parameters (`best_params`) are identified, I **retrain the final SVD++ model** using the combined `train_df` and `eval_df` datasets. This ensures the final model has seen as much data as possible, leading to better performance. During this process, I also update all global variables and data structures (`data_info`, `all_items`, `item_content_tfidf`, `pop_score`) to reflect the larger, combined dataset, which is a crucial step for consistent evaluation later on.



=== STEP 5: Hyperparameter Tuning for SVD++ (recommend_user-based eval) ===

>>> SVD++ trial: embed=32, lr=0.001, num_neg=3, epochs=10
Training start time: 2025-08-26 13:31:42


Eval SVDpp (top-20): 100%|██████████| 2000/2000 [00:01<00:00, 1971.80it/s]


Trial -> recall@20: 0.3486, ndcg@20: 0.2381

>>> SVD++ trial: embed=32, lr=0.001, num_neg=3, epochs=20
Training start time: 2025-08-26 14:20:06


Eval SVDpp (top-20): 100%|██████████| 2000/2000 [00:00<00:00, 2066.91it/s]


Trial -> recall@20: 0.3474, ndcg@20: 0.2288

>>> SVD++ trial: embed=32, lr=0.001, num_neg=5, epochs=10
Training start time: 2025-08-26 15:55:12


...

Trial -> recall@20: 0.3462, ndcg@20: 0.2301

>>> SVD++ trial: embed=64, lr=0.001, num_neg=3, epochs=20
Training start time: 2025-08-27 02:27:11


Eval SVDpp (top-20): 100%|██████████| 2000/2000 [00:00<00:00, 2121.91it/s]


Trial -> recall@20: 0.3331, ndcg@20: 0.2169

>>> SVD++ trial: embed=64, lr=0.001, num_neg=5, epochs=10
Training start time: 2025-08-27 04:36:03


Eval SVDpp (top-20): 100%|██████████| 2000/2000 [00:00<00:00, 2088.15it/s]


Trial -> recall@20: 0.3443, ndcg@20: 0.2281

>>> SVD++ trial: embed=64, lr=0.001, num_neg=5, epochs=20
Training start time: 2025-08-27 06:12:08


In [17]:
# -------------------- STEP 5b: Train ItemCF baseline --------------------
print("\n=== STEP 5b: Train ItemCF (ItemKNN) baseline ===")
itemcf = ItemCF(
    task="ranking",
    data_info=data_info,
    sim_type="cosine",
    k_sim=50,
    store_top_k=True,
    block_size=None,
    num_threads=CPU_CORES,
    min_common=1,
    mode="invert",
    seed=SEED,
)
itemcf.fit(train_data_final, neg_sampling=True, verbose=0)

# HERE I AM DOING: Training a second collaborative filtering model, ItemCF, as a baseline.
# I REPEAT: ItemCF (Item-based Collaborative Filtering) is a classic recommendation algorithm. I am training an `ItemCF` model from the `libreco` library on the same `train_data_final` as the SVD++ model. This model will serve as a strong, non-deep-learning baseline against which we can compare the performance of SVD++ and our final hybrid model.



=== STEP 5b: Train ItemCF (ItemKNN) baseline ===
Training start time: 2025-08-24 00:32:38
Final block size and num: (8184, 1)
sim_matrix elapsed: 2.225s
sim_matrix, shape: (8184, 8184), num_elements: 25085682, density: 37.4537 %


top_k: 100%|██████████| 8184/8184 [00:07<00:00, 1057.44it/s]


In [19]:
# -------------------- STEP 6: Build user content profiles (playtime-weighted) ----
print("\n=== STEP 6: Build user content profiles (playtime-weighted) ===")
# Use only interactions belonging to train_plus_eval_df for profile construction
wdf = train_plus_eval_df.merge(inter_df[['user','item','playtime']], on=['user','item'], how='left')
wdf['w'] = np.clip(np.log1p(wdf['playtime'].astype(float)), 0.0, 10.0)

user_item_w = defaultdict(list)
for _, r in wdf.iterrows():
    u, it, w = int(r['user']), int(r['item']), float(r['w'])
    pos = item2pos.get(it, None)
    if pos is not None:
        user_item_w[u].append((pos, w))

user_prof, user_prof_norm, user_tagged_counts = {}, {}, {}
for u in tqdm(all_users, desc="User profiles"):
    pairs = user_item_w.get(int(u), [])
    if not pairs:
        user_prof[int(u)] = None; user_prof_norm[int(u)] = 0.0; user_tagged_counts[int(u)] = 0
        continue
    idxs = np.array([p[0] for p in pairs], dtype=int)
    ws    = np.array([p[1] for p in pairs], dtype=float)
    sw    = ws.sum()
    user_tagged_counts[int(u)] = int((item_content_tfidf[idxs].getnnz(axis=1) > 0).sum())
    if sw <= 0:
        user_prof[int(u)] = None; user_prof_norm[int(u)] = 0.0; continue
    v = (item_content_tfidf[idxs].T.dot(ws) / (sw + 1e-12)).ravel()
    n = np.linalg.norm(v)
    if n > 0: v = v / n
    user_prof[int(u)] = v; user_prof_norm[int(u)] = float(n)

hist_counts = train_plus_eval_df.groupby('user')['item'].count().to_dict()

# HERE I AM DOING: Creating content profiles for each user based on their past interactions.
# I REPEAT: This is the critical step for the **content-based** part of the hybrid model.
# I am building a "user profile" vector for every user. This vector represents the user's content preferences. I do this by taking the TF-IDF vectors of all the items a user has interacted with and creating a weighted average. The weights are based on the user's `playtime` for each item. Specifically, I apply a `log` transformation to `playtime` to soften the impact of extremely long playtimes. This process results in a single vector per user that captures their content interests (e.g., they like "action_rpg" and "rogue_like" games). The resulting profiles are stored in the `user_prof` dictionary. I also store the L2 norm of the profile vector (`user_prof_norm`) and the number of tagged items a user interacted with (`user_tagged_counts`), which are both used later for the adaptive blending logic.



=== STEP 6: Build user content profiles (playtime-weighted) ===


User profiles: 100%|██████████| 53694/53694 [00:17<00:00, 3114.76it/s]


In [21]:
# -------------------- STEP 7: Build test truth --------------------
print("\n=== STEP 7: Build test truth ===")
test_truth = test_df.groupby('user')['item'].apply(list).to_dict()

# HERE I AM DOING: Preparing the ground-truth data for final evaluation.
# I REPEAT: I am creating a dictionary called `test_truth` from the `test_df` dataframe. This dictionary maps each user ID to a list of items they interacted with in the test set. This is the "gold standard" to which all model recommendations will be compared to calculate the final evaluation metrics.



=== STEP 7: Build test truth ===


In [23]:
# -------------------- Scoring helpers --------------------
def cf_scores_user(u, top_n=CAND_CF):
    if int(u) not in svdpp.data_info.user2id:
        return np.zeros(len(all_items), dtype=float)
    recs = svdpp.recommend_user(int(u), n_rec=top_n, filter_consumed=True, inner_id=False)
    if isinstance(recs, dict):
        arr = recs.get(int(u), np.array([], dtype=int))
    else:
        arr = np.asarray(recs)
    scores = np.zeros(len(all_items), dtype=float)
    for r, it in enumerate(arr[:top_n]):
        pos = item2pos.get(int(it), None)
        if pos is not None:
            scores[pos] = 1.0 / float(r + 1)
    return scores

def icf_scores_user(u, top_n=CAND_CF):
    if int(u) not in itemcf.data_info.user2id:
        return np.zeros(len(all_items), dtype=float)
    recs = itemcf.recommend_user(int(u), n_rec=top_n, filter_consumed=True, inner_id=False)
    if isinstance(recs, dict):
        arr = recs.get(int(u), np.array([], dtype=int))
    else:
        arr = np.asarray(recs)
    scores = np.zeros(len(all_items), dtype=float)
    for r, it in enumerate(arr[:top_n]):
        pos = item2pos.get(int(it), None)
        if pos is not None:
            scores[pos] = 1.0 / float(r + 1)
    return scores

def cb_scores_user(u):
    prof = user_prof.get(int(u))
    if prof is None: return np.zeros(len(all_items), dtype=float)
    s = item_content_tfidf.dot(prof)
    return np.asarray(s).ravel()

def pop_scores_user(u):
    return pop_score

# wrappers for eval rec_dist, plotting etc
def get_scores_svd(u): return cf_scores_user(u)
def get_scores_icf(u): return icf_scores_user(u)
def get_scores_cb(u):  return cb_scores_user(u)
def get_scores_pop(u): return pop_scores_user(u)
def get_scores_hybrid(u, base_alpha):
    a_u = conservative_alpha(u, base_alpha)
    cf_all = cf_scores_user(u)
    cb_all = cb_scores_user(u)
    return rrf_blend_from_scores(exclude_consumed_idx(u), cf_all, cb_all, a_u)

# HERE I AM DOING: Defining functions that generate scores for a given user from each of our different models.
# I REPEAT: These helper functions provide a standardized way to get a list of scores for all possible items for a given user.
# - `cf_scores_user`: Generates scores from the SVD++ model. It gets a list of recommendations from `svdpp.recommend_user` and converts them into a score vector where the top-ranked item gets the highest score (`1/1`), the second gets `1/2`, etc. Items not in the top-N receive a score of 0.
# - `icf_scores_user`: Does the same thing as `cf_scores_user` but for the ItemCF model.
# - `cb_scores_user`: Generates scores for the content-based model. It calculates the dot product between the user's content profile vector and the TF-IDF matrix of all items. The result is a score vector where a higher score indicates a better match between the user's profile and the item's content.
# - `pop_scores_user`: Simply returns the pre-calculated popularity scores for all items.
# - `get_scores_hybrid`: This is the most complex one. It combines the `cf_scores_user` and `cb_scores_user` results. It first calculates an adaptive `alpha` value for the specific user using `conservative_alpha`, then uses `rrf_blend_from_scores` to perform the weighted Reciprocal Rank Fusion of the two score vectors. This function encapsulates the entire hybrid logic.


In [25]:
# -------------------- STEP 8: Evaluate SVD / Content / Popularity / ItemCF / Hybrid --------------------
print("\n=== STEP 8: Evaluate SVD / Content / Popularity / ItemCF / Hybrid ===")

# Evaluate SVDpp using recommend_user-based evaluation (canonical)
svd_res_eval = recommend_eval_by_model_recommend_user(svdpp, test_truth, k=K, users_sample=None)
print(f"SVDpp -> recall@{K}: {svd_res_eval['recall']:.4f}, ndcg@{K}: {svd_res_eval['ndcg']:.4f}")

# Evaluate ItemCF similarly (canonical)
icf_res_eval = recommend_eval_by_model_recommend_user(itemcf, test_truth, k=K, users_sample=None)
print(f"ItemCF -> recall@{K}: {icf_res_eval['recall']:.4f}, ndcg@{K}: {icf_res_eval['ndcg']:.4f}")

# Evaluate Content & Popularity using score-based evaluator (consistent filtering)
cb_res_eval = eval_model_scores(cb_scores_user, "Content", k=K)
pop_res_eval = eval_model_scores(pop_scores_user, "Popularity", k=K)

# Build rows and ensure we keep diversity & pop_bias columns
rows = [
    {"model":"SVDpp", "recall":svd_res_eval["recall"], "ndcg":svd_res_eval["ndcg"], "hitrate":svd_res_eval["hitrate"], "coverage":svd_res_eval.get("coverage",0.0), "diversity":None, "pop_bias":None},
    {"model":"Content", "recall":cb_res_eval["recall"], "ndcg":cb_res_eval["ndcg"], "hitrate":cb_res_eval["hitrate"], "coverage":cb_res_eval["coverage"], "diversity":cb_res_eval["diversity"], "pop_bias":cb_res_eval["pop_bias"]},
    {"model":"Popularity", "recall":pop_res_eval["recall"], "ndcg":pop_res_eval["ndcg"], "hitrate":pop_res_eval["hitrate"], "coverage":pop_res_eval["coverage"], "diversity":pop_res_eval["diversity"], "pop_bias":pop_res_eval["pop_bias"]},
    {"model":"ItemCF", "recall":icf_res_eval["recall"], "ndcg":icf_res_eval["ndcg"], "hitrate":icf_res_eval["hitrate"], "coverage":icf_res_eval.get("coverage",0.0), "diversity":None, "pop_bias":None},
]

# Alpha-sweep for Hybrid (global alpha) using eval_model_scores wrapper
print("\nGlobal-α hybrid (RRF) sweep:")
hgrid = []
for a in ALPHAS_GRID:
    res = eval_model_scores(lambda u: rrf_blend_from_scores(exclude_consumed_idx(u), cf_scores_user(u), cb_scores_user(u), conservative_alpha(u, a)), f"Hybrid(a={a:.2f})", k=K)
    print(f"alpha={a:.2f} -> recall@{K}: {res['recall']:.4f}, ndcg@{K}: {res['ndcg']:.4f}")
    res["alpha"] = float(a)
    hgrid.append(res)

# Adaptive hybrid (base_alpha=0.5)
had_res = eval_model_scores(lambda u: rrf_blend_from_scores(exclude_consumed_idx(u), cf_scores_user(u), cb_scores_user(u), conservative_alpha(u, 0.5)), "Hybrid-Adapt", k=K)
rows.append({"model":"Hybrid-Adapt", "recall":had_res["recall"], "ndcg":had_res["ndcg"], "hitrate":had_res["hitrate"], "coverage":had_res["coverage"], "diversity":had_res["diversity"], "pop_bias":had_res["pop_bias"]})

summary_df = pd.DataFrame(rows)[["model","recall","ndcg","hitrate","coverage","diversity","pop_bias"]]
summary_df = summary_df.sort_values(["recall","ndcg"], ascending=False).reset_index(drop=True)
print("\nEvaluation summary:\n", summary_df)
summary_df.to_csv(EVAL_SUMMARY_OUT, index=False)
print(f"Saved evaluation summary to {EVAL_SUMMARY_OUT}")

# HERE I AM DOING: The final evaluation of all models against the held-out test set.
# I REPEAT: This is where I measure the performance of each model.
# - I first evaluate the SVD++ and ItemCF models using `recommend_eval_by_model_recommend_user`. This is the canonical way to evaluate these `libreco` models.
# - Then, I evaluate the Content and Popularity baselines using the more flexible `eval_model_scores` function, which also calculates diversity and popularity bias.
# - I perform an **alpha-sweep** for the hybrid model. I loop through all the `ALPHAS_GRID` values and evaluate the hybrid model's performance for each alpha. This helps identify the optimal balance between CF and CB.
# - Finally, I evaluate the **adaptive hybrid model**, where the alpha value is adjusted per user.
# All the results are collected into a pandas DataFrame (`summary_df`), which is then sorted and saved as a CSV file. This provides a clean, side-by-side comparison of all the models on all the metrics.



=== STEP 8: Evaluate SVD / Content / Popularity / ItemCF / Hybrid ===


Eval SVDpp (top-20): 100%|██████████| 42071/42071 [00:15<00:00, 2678.48it/s]


SVDpp -> recall@20: 0.3706, ndcg@20: 0.2666


Eval ItemCF (top-20): 100%|██████████| 42071/42071 [09:00<00:00, 77.87it/s] 


ItemCF -> recall@20: 0.3103, ndcg@20: 0.2332


...

alpha=0.90 -> recall@20: 0.3665, ndcg@20: 0.2428


Eval scores Hybrid(a=0.95) (top-20): 100%|██████████| 42071/42071 [07:27<00:00, 93.95it/s] 


alpha=0.95 -> recall@20: 0.3690, ndcg@20: 0.2490


Eval scores Hybrid-Adapt (top-20): 100%|██████████| 42071/42071 [07:37<00:00, 91.96it/s] 



Evaluation summary:
           model    recall      ndcg   hitrate  coverage  diversity  pop_bias
0         SVDpp  0.370556  0.266606  0.759240  0.210166        NaN       NaN
1        ItemCF  0.310340  0.233194  0.697155  0.155425        NaN       NaN
2    Popularity  0.272171  0.202940  0.643317  0.010386   0.735618  0.998047
3  Hybrid-Adapt  0.228777  0.146453  0.585605  0.313661   0.573602  0.963367
4       Content  0.088262  0.051652  0.293599  0.439027   0.521658  0.904107
Saved evaluation summary to C:\Users\kerne\Pictures\datasets\evaluation_summary.csv


In [27]:
# -------------------- STEP 10: Result plots --------------------
# A) Accuracy bars
plt.figure(figsize=(9,5))
x = np.arange(len(summary_df))
plt.bar(x-0.2, summary_df["recall"].values, width=0.4, label=f"Recall@{K}")
plt.bar(x+0.2, summary_df["ndcg"].values, width=0.4, label=f"NDCG@{K}")
plt.xticks(x, summary_df["model"].values, rotation=15, ha="right")
plt.ylabel("Score"); plt.title("Top-K Ranking Quality"); plt.legend()
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "res_accuracy_bars.png")); plt.close()

# B) Coverage & Diversity
plt.figure(figsize=(9,5))
plt.bar(x-0.2, summary_df["coverage"].values, width=0.4, label=f"Catalog Coverage@{K}")
plt.bar(x+0.2, summary_df["diversity"].fillna(0).values, width=0.4, label=f"Intra-List Diversity@{K}")
plt.xticks(x, summary_df["model"].values, rotation=15, ha="right")
plt.ylabel("Score"); plt.title("Coverage & Diversity Trade-offs"); plt.legend()
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "res_coverage_diversity_bars.png")); plt.close()

# C) Popularity bias
plt.figure(figsize=(9,5))
plt.bar(x, summary_df["pop_bias"].fillna(0).values)
plt.xticks(x, summary_df["model"].values, rotation=15, ha="right")
plt.ylabel("Avg popularity percentile (higher=>head)"); plt.title("Popularity Bias of Recommendations")
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "res_pop_bias.png")); plt.close()

# D) alpha sweep curves
if hgrid:
    dfh = pd.DataFrame(hgrid)
    plt.figure(figsize=(8,5))
    plt.plot(dfh["alpha"].values, dfh["recall"].values, marker='o', label=f"Recall@{K}")
    plt.plot(dfh["alpha"].values, dfh["ndcg"].values, marker='s', label=f"NDCG@{K}")
    plt.xlabel("Alpha (CF weight)"); plt.ylabel("Score")
    plt.title("Hybrid α-sweep"); plt.grid(True); plt.legend()
    plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "res_hybrid_alpha_sweep.png")); plt.close()

print("Saved plots.")

# HERE I AM DOING: Creating visualizations of the evaluation results.
# I REPEAT: I am using `matplotlib` to generate four distinct plots to easily compare the performance of the different models.
# - **Accuracy bars**: This plot shows `Recall@K` and `NDCG@K` side-by-side for each model, making it easy to see which model is the most accurate at finding relevant items and ranking them correctly.
# - **Coverage & Diversity**: This plot compares the `Catalog Coverage` (how many unique items are recommended across all users) and `Intra-List Diversity` (how different the items in a single user's list are) for each model. This is key for understanding trade-offs beyond simple accuracy.
# - **Popularity bias**: This plot shows the average popularity of the recommended items. A model with high popularity bias tends to recommend only popular items, which can be bad for discoverability.
# - **Alpha sweep curves**: This plot visualizes how the hybrid model's accuracy changes as the `alpha` value (the weight given to the CF model) is varied. This helps confirm the optimal blending ratio.
# All plots are saved as PNG files for easy sharing and analysis.



Saved plots.


In [ ]:
# Find best alpha (highest NDCG) for hybrid
df_hybrid = pd.DataFrame(hgrid)
best_idx = df_hybrid["ndcg"].idxmax()
best_alpha_res = df_hybrid.loc[best_idx]

# Collect all final models for comparison
models_to_plot = [
    {"model": "SVDpp", "recall": svd_res_eval["recall"], "ndcg": svd_res_eval["ndcg"]},
    {"model": "ItemCF", "recall": icf_res_eval["recall"], "ndcg": icf_res_eval["ndcg"]},
    {"model": "Content", "recall": cb_res_eval["recall"], "ndcg": cb_res_eval["ndcg"]},
    {"model": "Popularity", "recall": pop_res_eval["recall"], "ndcg": pop_res_eval["ndcg"]},
    {"model": f"Hybrid_best_alpha ({best_alpha_res['alpha']:.2f})",
     "recall": best_alpha_res["recall"],
     "ndcg": best_alpha_res["ndcg"]}
]

df_plot = pd.DataFrame(models_to_plot)

plt.figure(figsize=(10,5))
x = np.arange(len(df_plot))
plt.bar(x-0.2, df_plot["recall"], width=0.4, label="Recall@20")
plt.bar(x+0.2, df_plot["ndcg"], width=0.4, label="NDCG@20")
plt.xticks(x, df_plot["model"], rotation=15, ha="right")
plt.ylabel("Score")
plt.title("Top-K Ranking Comparison (Best Hybrid α vs Baselines)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "best_comparison_recall_ndcg.png"))
plt.show()


In [ ]:
df_hybrid_metrics = df_hybrid[["alpha", "recall", "ndcg"]].copy()
csv_out = os.path.join(DATA_DIR, "hybrid_alpha_metrics.csv")
df_hybrid_metrics.to_csv(csv_out, index=False)
print(f"Saved hybrid alpha metrics to {csv_out}")

# Optional: quick line plot of the sweep
plt.figure(figsize=(8,5))
plt.plot(df_hybrid_metrics["alpha"], df_hybrid_metrics["recall"], marker='o', label="Recall@20")
plt.plot(df_hybrid_metrics["alpha"], df_hybrid_metrics["ndcg"], marker='s', label="NDCG@20")
plt.xlabel("Alpha (CF weight)")
plt.ylabel("Score")
plt.title("Hybrid Alpha Sweep Metrics")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "hybrid_alpha_sweep_metrics.png"))
plt.show()


In [29]:
# -------------------- STEP 11: Generate top-20 for test_interactions_in.csv --------------------
print("\n=== STEP 11: Generate top-20 for test_interactions_in.csv (best global α) ===")
if test_in_df is None:
    print("No test_interactions_in.csv found; skipping final recommendations.")
else:
    if hgrid:
        dfh = pd.DataFrame(hgrid)
        # pick best by recall then ndcg
        best_alpha = float(dfh.sort_values(["recall","ndcg"], ascending=False).iloc[0]["alpha"])
        print(f"Choosing best_alpha from sweep: {best_alpha:.2f}")
    else:
        best_alpha = 0.95

    out_rows = []
    for u in tqdm(test_in_df["user"].unique(), desc="Inferring"):
        allow = exclude_consumed_idx(u)
        a_u = conservative_alpha(u, best_alpha)
        cf_all = cf_scores_user(u)
        cb_all = cb_scores_user(u)
        scores = rrf_blend_from_scores(allow, cf_all, cb_all, a_u)
        idxs = topk_from_scores(scores, allow, K)
        for pos in idxs:
            out_rows.append((int(u), int(all_items[pos])))
    out_df = pd.DataFrame(out_rows, columns=["user_id","item_id"])
    out_df.to_csv(OUTPUT_TOP20, index=False)
    print(f"Saved top-{K} recommendations to {OUTPUT_TOP20} ({len(out_df):,} rows)")


=== STEP 11: Generate top-20 for test_interactions_in.csv (best global α) ===
Choosing best_alpha from sweep: 0.95


Inferring: 100%|██████████| 13579/13579 [00:28<00:00, 475.97it/s]


Saved top-20 recommendations to C:\Users\kerne\Pictures\datasets\top20_recommendations.csv (271,580 rows)
